# echem_tools — Colab 분석 노트북
EIS→DRT / 고전압 플로팅 누설전류 분석. **순서대로 셀을 실행**하세요.

1. 설치  2. 파일 업로드  3. 설정 선택  4. 실행 → 그림·표 표시 + `report.html`(및 전체 결과 zip) 다운로드

지원 형식: BioLogic `.mpt`, Gamry `.DTA`, CSV(일반/Neware/LANHE, 열 이름은 `config.yaml`의 `csv_profiles`로 매핑).
플로팅 그룹 비교를 하려면 `metadata.csv`(cell_id, file, type, surface, electrolyte, active_mass_mg, area_cm2, note)도 함께 업로드하세요.
`file` 열에는 업로드한 파일 이름만 적으면 됩니다.

In [ ]:
#@title 1. 설치 (약 1분)
import os, subprocess, sys
REPO_URL = "https://github.com/wj7294-blip/Calculation.git"  #@param {type:"string"}
SUBDIR = "echem_tools"  #@param {type:"string"}
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/repo"], check=True)
TOOL_DIR = os.path.join("/content/repo", SUBDIR)
%cd {TOOL_DIR}
!pip -q install -r requirements.txt
sys.path.insert(0, TOOL_DIR)
print("ready:", TOOL_DIR)

In [ ]:
#@title 2. 데이터 파일 업로드 (여러 개 선택 가능; metadata.csv 포함 가능)
from google.colab import files
import shutil, pathlib
UP = pathlib.Path("/content/uploads"); UP.mkdir(exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, UP / name)
print("uploaded:", sorted(p.name for p in UP.iterdir()))

In [ ]:
#@title 3. 분석 종류와 설정값
analysis = "DRT"  #@param ["DRT", "Floating"]
#@markdown **DRT 설정**
drt_lambda = "auto"  #@param {type:"string"}
inductive_tail = "fit"  #@param ["fit", "drop", "keep"]
freq_min_Hz = 0.01  #@param {type:"number"}
freq_max_Hz = 1000000.0  #@param {type:"number"}
#@markdown **플로팅 설정**
transient_h = 1.0  #@param {type:"number"}
leak_window_h = 1.0  #@param {type:"number"}
V_hold = 4.6  #@param {type:"number"}
#@markdown metadata.csv 없이 단일 셀만 볼 때 사용 (metadata가 있으면 무시):
active_mass_mg = 8.0  #@param {type:"number"}
area_cm2 = 1.13  #@param {type:"number"}
#@markdown CSV 열 매핑 프로파일 (CSV 입력일 때)
csv_profile = "generic"  #@param ["generic", "neware", "lanhe"]
print("analysis =", analysis)

In [ ]:
#@title 4. 실행 → 결과 표시 → 다운로드
import sys, pathlib, shutil, glob
sys.path.insert(0, "/content/repo/echem_tools")
import importlib, analyze; importlib.reload(analyze)
from IPython.display import display, Image, HTML
import pandas as pd
UP = pathlib.Path("/content/uploads"); OUT = pathlib.Path("/content/results")
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir()
meta = UP / "metadata.csv"
data_files = [str(p) for p in sorted(UP.iterdir()) if p.name.lower() != "metadata.csv"]
if analysis == "DRT":
    args = ["drt", *data_files, "--out", str(OUT), "--lambda", str(drt_lambda), "--inductive", inductive_tail,
            "--fmin", str(freq_min_Hz), "--fmax", str(freq_max_Hz), "--profile", csv_profile]
    if meta.exists(): args += ["--meta", str(meta), "--data-dir", str(UP)]
else:
    args = ["floating", "--out", str(OUT), "--transient-h", str(transient_h), "--leak-window-h", str(leak_window_h),
            "--v-hold", str(V_hold), "--profile", csv_profile]
    if meta.exists(): args += ["--meta", str(meta), "--data-dir", str(UP)]
    else: args += data_files + ["--mass-mg", str(active_mass_mg), "--area-cm2", str(area_cm2)]
print("analyze.py", " ".join(args))
analyze.main(args)
# ---- show tables & figures inline
for csv in ["eis_summary.csv", "eis_peaks.csv", "float_cells.csv", "float_group_summary.csv", "float_anova.csv"]:
    p = OUT / csv
    if p.exists():
        display(HTML(f"<h3>{csv}</h3>")); display(pd.read_csv(p))
for png in sorted(glob.glob(str(OUT / "**" / "*.png"), recursive=True)):
    if any(k in png for k in ["compare_", "interaction_", "group_", "all_cells", "_drt.png"]):
        display(Image(filename=png, width=1000))
# ---- download
from google.colab import files
shutil.make_archive("/content/results", "zip", OUT)
files.download(str(OUT / "report.html"))
files.download("/content/results.zip")

### 참고
- `report.html` 하나에 모든 그림·표가 포함되어 있습니다(이미지 내장). zip 에는 CSV/PNG 원본이 들어 있습니다.
- DRT λ: `auto`(GCV)가 기본. 피크가 너무 갈라지면 λ를 키우고(예: `1e-2`), 뭉개지면 줄이세요(예: `1e-5`). 영향은 README 의 "λ 선택" 절 참조.
- 플로팅: 정전압 구간은 `V_hold ± V_tol`(config.yaml)로 자동 검출하고, Neware/LANHE CSV 는 step 열의 CV 라벨도 사용합니다.